In [0]:
%run ./00_config

In [0]:
from pyspark.sql.functions import col, count, when, isnan

def profile_dataset(name, path, format_opts={"header": "true", "inferSchema": "true"}):
    print(f"\n====================================================================")
    print(f"📊 PROFILING DATASET: {name}")
    print(f"====================================================================")
    try:
        # Load data using standard CSV reader
        df = spark.read.format("csv").options(**format_opts).load(path)
        
        # 1. Structural Metric Extractions (Fulfills Your Core Request)
        row_count = df.count()
        col_names = df.columns
        col_count = len(col_names)
        
        print(f"📈 Row Count    : {row_count}")
        print(f"📐 Column Count : {col_count}")
        print(f"📋 Column Names : {', '.join(col_names)}")
        
        # 2. Defect Scan: Check for Null / Missing Values
        print("\n❌ DEFECT REPORT: Null/Missing Values Per Column:")
        null_counts = df.select([count(when(col(c).isNull() | isnan(col(c)) | (col(c) == ""), c)).alias(c) for c in col_names]).collect()[0]
        for col_name in col_names:
            print(f"  └─ {col_name}: {null_counts[col_name]} null/missing records")
                
    except Exception as e:
        print(f"⚠️ Error profiling {name}: {str(e)}")

# Execute comprehensive profile runs across all 5 distinct targets
profile_dataset("Customers", path_customers)
profile_dataset("Accounts", path_accounts)
profile_dataset("Branches", path_branches)
profile_dataset("Transactions (Batch 1 Ingestion)", path_transactions)
profile_dataset("Transactions (Batch 2 Staging)", f"{volume_root_path}/staging/transactions_batch_02.csv")
